In [1]:
from pymilvus import Collection, FieldSchema, CollectionSchema, DataType, connections, utility

connections.connect(
  alias="default",
  host='localhost',
  port='19530'
)

collection = Collection("Album1")
collection.compact()
utility.list_collections()

['Album1']

In [2]:
# Load the collection in to the memory
collection.load(replica_number=1)

In [3]:
## Vector Similarity Search

results = collection.search(
	data=[[0.1, 0.2]], 
	anns_field="song_vec", 
	param={"metric_type": "L2", "params": {"search_k": 64}},
	limit=5, 
	expr=None,
	output_fields=['song_name']
)

for result in results[0]:
    print (result)

id: 1, distance: 0.03780278563499451, entity: {'song_name': 'BALORHM'}
id: 3, distance: 0.23100890219211578, entity: {'song_name': 'YAIJTYF'}
id: 2, distance: 0.32868140935897827, entity: {'song_name': 'BIUQUGN'}
id: 4, distance: 1.2851006984710693, entity: {'song_name': 'AVTLGAE'}


In [4]:
# Query the data in scalar field

query_res = collection.query(
  expr = "song_id in [1,2]",
  limit = 10, 
  output_fields = ["song_name", "listen_count"]
)

for result in query_res:
    print (result)

{'song_name': 'BALORHM', 'listen_count': 74074, 'song_id': 1}
{'song_name': 'BIUQUGN', 'listen_count': 84576, 'song_id': 2}


In [5]:
# Hybrid search

hybrid_res = collection.search(
	data=[[0.1, 0.2]], 
	anns_field="song_vec", 
	param={"metric_type": "L2", "params": {"search_k": 64}},
	limit=5, 
	expr="listen_count <= 100000",
	output_fields=['song_name']
)

for result in hybrid_res[0]:
    print (result)

id: 1, distance: 0.03780278563499451, entity: {'song_name': 'BALORHM'}
id: 3, distance: 0.23100890219211578, entity: {'song_name': 'YAIJTYF'}
id: 2, distance: 0.32868140935897827, entity: {'song_name': 'BIUQUGN'}
id: 4, distance: 1.2851006984710693, entity: {'song_name': 'AVTLGAE'}


In [6]:
# Release the collection from the memory
collection.release()